# AFT-030: Circular Delegation -- Demo**Class:** Supervisor Coordination | **Severity:** P1Demonstrates how a multi-agent supervisor enters a delegation cycle andhow cycle detection (not depth limits) resolves it.

In [ ]:
from dataclasses import dataclass@dataclassclass Agent:    name: str    redirect_to: str | None = None    def handle(self, query):        if self.redirect_to:            return {"status":"redirect","target":self.redirect_to}        return {"status":"success","data":f"{self.name}: {query}"}agents = {    "customer": Agent("customer", redirect_to="pricing"),    "pricing": Agent("pricing", redirect_to="customer"),    "fulfillment": Agent("fulfillment", redirect_to=None),}print("Topology: customer -> pricing -> customer (cycle!)")print("fulfillment handles requests directly")

## The Failure: Uncontrolled Cycling

In [ ]:
class NaiveSupervisor:    def __init__(self, agents, max_depth=10):        self.agents = agents        self.max_depth = max_depth        self.steps = []    def delegate(self, query, target, depth=0):        if depth >= self.max_depth:            return {"status":"error","msg":"max depth"}        r = self.agents[target].handle(query)        self.steps.append(target)        if r["status"]=="redirect":            return self.delegate(query, r["target"], depth+1)        return rnaive = NaiveSupervisor(agents, 10)r = naive.delegate("Acme renewal price?","customer")print(f"Result: {r['status']}")print(f"Steps ({len(naive.steps)}): {' -> '.join(naive.steps)}")print(f"All {len(naive.steps)} steps wasted. ~{len(naive.steps)*2}s latency in production.")

## Wrong Fix: Depth Limit = 3Stops cycles but also breaks legitimate chains.

In [ ]:
legit = {    "router": Agent("router", redirect_to="customer"),    "customer": Agent("customer", redirect_to="pricing"),    "pricing": Agent("pricing", redirect_to="fulfillment"),    "fulfillment": Agent("fulfillment", redirect_to=None),}dl = NaiveSupervisor(legit, 3)r = dl.delegate("Process order","router")print(f"Depth limit 3 on 4-step chain: {r['status']}")print(f"Steps: {' -> '.join(dl.steps)}")print("BROKEN: legitimate chain exceeded depth limit")

## Correct Fix: Cycle DetectionTrack visited agents. Cycle = revisiting an agent already in the chain.

In [ ]:
class CircularDelegationError(Exception):    def __init__(self, cycle): self.cycle = cycle; super().__init__(str(cycle))class CycleAwareSupervisor:    def __init__(self, agents):        self.agents = agents        self.steps = []    def delegate(self, query, target, chain=None):        if chain is None: chain = []        if target in chain:            idx = chain.index(target)            raise CircularDelegationError(chain[idx:]+[target])        chain.append(target)        self.steps.append(target)        r = self.agents[target].handle(query)        if r["status"]=="redirect":            return self.delegate(query, r["target"], chain)        return r# Test 1: catches cycleprint("Test 1: Circular topology")try:    CycleAwareSupervisor(agents).delegate("Acme price?","customer")except CircularDelegationError as e:    print(f"  Caught cycle: {' -> '.join(e.cycle)}")# Test 2: allows legitimate chainprint("\nTest 2: Legitimate 4-step chain")ca = CycleAwareSupervisor(legit)r = ca.delegate("Process order","router")print(f"  Result: {r['status']}")print(f"  Chain: {' -> '.join(ca.steps)}")print("  No false positive.")

## Key TakeawayDepth limits != cycle detection. A depth limit of N blocks bothN-step legitimate chains and N-step cycles. Cycle detection blocksonly actual cycles, at any depth.**Detection:** Same `agent_id` appearing 2x in a request's delegation chain.